# UniMER : reconnaissance multilingue et multimodale des émotions

**Article :** Qin & Anastasiu, *UniMER: Multilingual Multimodal Emotion Recognition with Prototype-Guided Attention Pooling*, EMNLP 2026.

## Objectifs pédagogiques
1. Expliquer pourquoi l'évaluation sur des données déséquilibrées peut tromper.
2. Comprendre l'idée du langage naturel comme interface universelle.
3. Calculer le nombre de paramètres d'un adaptateur LoRA.
4. Implémenter le pooling d'attention par prototypes de classe.
5. Analyser des résultats et des ablations de façon critique.

## Plan (environ 90 minutes)
| Partie | Thème | Durée |
|---|---|---|
| 1 | Contexte et déséquilibre des classes | 15 min |
| 2 | Idée centrale : le langage comme interface | 10 min |
| 3 | Architecture : LoRA, pooling par prototypes, pondération | 25 min |
| 4 | Résultats et ablations | 20 min |
| 5 | Analyse critique et exercice | 20 min |

Pour exécuter les cellules de code, il faut `numpy`, `matplotlib` et `torch`.

## Partie 1 : le problème du déséquilibre

Dans MELD, le neutre représente environ 47 % des énoncés. Un modèle qui répond toujours « neutre » obtient donc environ 47 % d'exactitude sans rien comprendre aux émotions. Calculons cela à partir des effectifs du tableau 2 de l'article.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Effectifs totaux de MELD (tableau 2 de l'article)
meld = {
    "Neutral": 6436, "Joy": 2308, "Surprise": 1636, "Anger": 1607,
    "Sadness": 1002, "Disgust": 361, "Fear": 358,
}
total = sum(meld.values())
print(f"Total : {total} énoncés")

# Baseline : toujours prédire la classe majoritaire
baseline_acc = meld["Neutral"] / total
print(f"Exactitude de la baseline « toujours neutre » : {baseline_acc:.1%}")
print(f"Rapport de déséquilibre (max/min) : {max(meld.values())/min(meld.values()):.1f} : 1")

plt.figure(figsize=(8, 4))
plt.bar(meld.keys(), meld.values(), color=["#4C72B0" if k not in ("Fear", "Disgust") else "#DD8452" for k in meld])
plt.ylabel("Nombre d'énoncés")
plt.title("Distribution des émotions dans MELD")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**Question pour les étudiants.** Si vous deviez noter un modèle sur MELD, quelle métrique utiliseriez-vous pour que la baseline « toujours neutre » ne paraisse pas bonne ? Réponse attendue : la macro-F1, qui donne le même poids à chaque classe.

Calculons un exemple pour le voir concrètement.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
# Pour éviter une dépendance supplémentaire, implémentation simple de la macro-F1
def macro_f1(y_true, y_pred, labels):
    scores = []
    for c in labels:
        tp = sum((t == c) and (p == c) for t, p in zip(y_true, y_pred))
        fp = sum((t != c) and (p == c) for t, p in zip(y_true, y_pred))
        fn = sum((t == c) and (p != c) for t, p in zip(y_true, y_pred))
        prec = tp / (tp + fp) if tp + fp else 0
        rec = tp / (tp + fn) if tp + fn else 0
        scores.append(2 * prec * rec / (prec + rec) if prec + rec else 0)
    return np.mean(scores)

labels = list(range(7))
# Simulation : 1000 énoncés avec la distribution de MELD, prédiction « toujours neutre » (classe 0)
rng = np.random.default_rng(0)
probs = np.array(list(meld.values())) / total
y_true = rng.choice(7, size=1000, p=probs)
y_pred = np.zeros(1000, dtype=int)
print(f"Exactitude : {accuracy_score(y_true, y_pred):.1%}")
print(f"Macro-F1   : {macro_f1(y_true, y_pred, labels):.1%}")

L'exactitude paraît raisonnable, alors que la macro-F1 est proche de 10 % : le modèle ne reconnaît aucune émotion rare.

## Partie 2 : le langage naturel comme interface universelle

Le principe est le suivant. Un LLM multimodal transforme la vidéo, le son et les expressions faciales en une description textuelle. Un seul modèle textuel traite ensuite le tout. Le tableau 1 de l'article en donne l'exemple : le texte « Look, we have enough, just walk away » semble calme, alors que la description visuelle (sourcils tendus, expressions inquiètes) révèle la peur.

Dans l'article, la fusion multimodale devient une simple concaténation de chaînes de caractères :

In [ ]:
context = "[CTX] Pourquoi tu dis ça ? [UTT] Look, we have enough, just walk away."
summary = ("[MMS] Voix tendue, hausse du ton. Sourcils fortement relevés, "
           "groupe aux expressions inquiètes. Ambiance de tension.")
model_input = f"{context} {summary} [SEP]"
print(model_input)
print()
print(f"Longueur en mots : {len(model_input.split())}")

**Discussion.**
- Avantage : pas besoin d'architecture de fusion différente pour chaque nouvelle modalité ou langue.
- Risque : la description est produite par un autre modèle. Si elle est fausse, le classifieur hérite de l'erreur.
- Question : quelles erreurs la description peut-elle introduire, par exemple un biais culturel dans l'interprétation des expressions faciales ?

## Partie 3 : architecture

### 3.1 LoRA : apprendre une petite correction

Au lieu de réentraîner une matrice de poids complète $W_0$, LoRA apprend une correction de rang faible $\Delta W = BA$, avec $B \in \mathbb{R}^{d \times r}$ et $A \in \mathbb{R}^{r \times k}$. Le nombre de paramètres passe de $d \times k$ à $r(d + k)$.

In [ ]:
d = k = 3840   # taille cachée de Gemma-3-12B (tableau 10)
r = 64         # rang LoRA (tableau 10)

full = d * k
lora = r * (d + k)
print(f"Une matrice complète : {full:,} paramètres")
print(f"Adaptateur LoRA     : {lora:,} paramètres")
print(f"Ratio               : {lora / full:.2%}")

Cette cellule ne reproduit pas exactement le total de 292 M de l'article, car les couches FFN n'ont pas la même forme. Elle montre en revanche le principe : pour une matrice carrée, LoRA ne garde qu'environ 3 % des paramètres.

### 3.2 Pooling par prototypes de classe

Le problème est que, dans un pooling d'attention classique, toutes les têtes apprennent les mêmes tokens saillants, souvent liés aux émotions majoritaires. L'article attribue à chaque classe une tête dédiée, pilotée par un vecteur prototype $p_k$ :

$$e^{(k)}_t = p_k^\top \tanh(W_h h_t + b_h), \quad \alpha^{(k)}_t = \frac{\exp(e^{(k)}_t)}{\sum_j \exp(e^{(k)}_j)}, \quad c_k = \sum_t \alpha^{(k)}_t h_t$$

Les $K$ vecteurs de contexte sont concaténés puis projetés (équation 6). Voici une implémentation minimale.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PrototypeAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, num_classes, proj_dim=256, out_dim=None, dropout=0.3):
        super().__init__()
        self.K = num_classes
        self.W_h = nn.Linear(hidden_dim, proj_dim)
        self.prototypes = nn.Parameter(torch.randn(num_classes, proj_dim) * 0.02)
        out_dim = out_dim or hidden_dim
        self.mlp = nn.Sequential(
            nn.Linear(num_classes * hidden_dim, out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, H):
        # H : (B, T, d)
        proj = torch.tanh(self.W_h(H))                       # (B, T, p)
        energy = torch.einsum("btp,kp->bkt", proj, self.prototypes)  # (B, K, T)
        alpha = F.softmax(energy, dim=-1)                    # poids par classe
        context = torch.einsum("bkt,btd->bkd", alpha, H)     # (B, K, d)
        pooled = self.mlp(context.flatten(1))                # (B, out_dim)
        return pooled, alpha

# Test avec des états cachés aléatoires
torch.manual_seed(0)
B, T, d, K = 2, 12, 64, 7
H = torch.randn(B, T, d)
pool = PrototypeAttentionPooling(d, K, proj_dim=32, out_dim=64)
s, alpha = pool(H)
print("Sortie poolée :", s.shape)           # (2, 64)
print("Poids d'attention :", alpha.shape)   # (2, 7, 12)
print("Somme des poids par classe :", alpha.sum(-1)[0])  # chaque ligne vaut 1

**Exercice rapide.** Remplacez les prototypes par un vecteur identique pour toutes les classes. Que se passe-t-il pour les poids $\alpha^{(k)}$ ? Ce cas illustre l'effondrement des têtes que l'article cherche à éviter.

In [ ]:
with torch.no_grad():
    pool.prototypes.copy_(torch.ones(K, 32) * 0.1)
_, alpha_collapsed = pool(H)
print("Écart entre les têtes (sans prototypes distincts) :",
      (alpha_collapsed[0, 0] - alpha_collapsed[0, 1]).abs().max().item())

### 3.3 Pondération des classes et perte

L'article utilise le « nombre effectif d'exemples » (Cui et al., 2019) : $w_k = (1-\beta)/(1-\beta^{n_k})$ avec $\beta = 0{,}9999$. Calculons les poids pour MELD.

In [ ]:
beta = 0.9999
counts = np.array(list(meld.values()))
names = list(meld.keys())
w = (1 - beta) / (1 - beta ** counts)
w = w / w.mean()   # normalisation pour garder une échelle comparable

for n, c, wi in zip(names, counts, w):
    print(f"{n:10s} n={c:5d}  poids={wi:.2f}")

Les émotions rares reçoivent un poids plus élevé. La perte totale combine la cross-entropie pondérée, un lissage des étiquettes ($\epsilon = 0{,}15$) et une régularisation KL (équation 7) : $\mathcal{L} = \mathcal{L}_{CE} + \lambda \mathcal{L}_{KL}$, avec $\lambda = 0{,}02$.

**Question.** Pourquoi les auteurs ajoutent-ils une régularisation si la pondération existe déjà ? Réponse attendue : la pondération corrige le déséquilibre global, mais la régularisation KL limite le risque qu'un petit nombre de classes capte toute la probabilité.

## Partie 4 : résultats et ablations

### 4.1 Émotions rares sur MELD (figure 2)
Les valeurs ci-dessous sont celles qui figurent dans le texte de l'article.

In [ ]:
emotions = ["Peur", "Dégoût"]
best_prior = [10.8, 10.7]
unimer = [39.6, 48.1]

x = np.arange(len(emotions))
plt.figure(figsize=(6, 4))
plt.bar(x - 0.2, best_prior, 0.4, label="Meilleure méthode antérieure")
plt.bar(x + 0.2, unimer, 0.4, label="UniMER")
plt.xticks(x, emotions)
plt.ylabel("F1 (%)")
plt.title("Émotions rares sur MELD")
plt.legend()
for i, (a, b) in enumerate(zip(best_prior, unimer)):
    plt.text(i - 0.2, a + 1, f"{a}", ha="center")
    plt.text(i + 0.2, b + 1, f"{b}", ha="center")
plt.tight_layout()
plt.show()

for e, a, b in zip(emotions, best_prior, unimer):
    print(f"{e} : x{b/a:.1f}")

### 4.2 Lire une ablation (tableau 7)

Le tableau 7 retire un composant à la fois. Calculons la contribution de chacun sur AkaCE, où l'effet est le plus marqué.

In [ ]:
# Macro-F1 sur AkaCE (tableau 7, graine 42)
full_system = 45.0
ablations = {
    "Sans pooling par prototypes": 41.0,
    "Sans captioning multimodal": 26.9,
    "Sans fine-tuning LoRA": 38.0,
}
for name, score in ablations.items():
    print(f"{name:30s} : perte de {full_system - score:.1f} pp")

**Question.** Le captioning multimodal fait perdre 14,1 pp sur AkaCE, mais seulement 2,7 pp sur MELD. Pourquoi ? Réponse attendue : en akan, les données textuelles sont limitées, et la prosodie et les expressions apportent davantage d'information.

## Partie 5 : analyse critique

Discutez les points suivants avec les étudiants.

1. **Petits effectifs de test.** Sur AkaCE, le dégoût ne compte que 11 exemples de test. Un seul exemple change le F1 de plusieurs points.
2. **Choix de la graine.** Les scores par classe sont ceux de la meilleure graine. Les moyennes sur cinq graines sont plus fiables, même si elles sont plus basses.
3. **Comparaisons.** Les méthodes antérieures sont reprises à partir de leurs chiffres publiés, et non réentraînées dans les mêmes conditions.
4. **Absence de vidéo sur M3ED.** L'évaluation chinoise est faite en texte seul, ce qui rend la comparaison multimodale incomplète.
5. **Éthique.** La détection d'émotions soulève des questions de vie privée, de consentement et de biais démographiques, que l'article mentionne lui-même.

### Exercice

Vous disposez des scores suivants de macro-F1 sur cinq graines pour une méthode donnée. Calculez la moyenne et l'écart-type, puis dites si la différence avec une autre méthode (moyenne 55,0) est significative.

In [ ]:
# Scores fictifs à utiliser pour l'exercice
scores_methode_a = [57.3, 55.8, 58.1, 56.2, 57.0]

mean_a = np.mean(scores_methode_a)
std_a = np.std(scores_methode_a, ddof=1)
print(f"Méthode A : {mean_a:.2f} ± {std_a:.2f}")
print(f"Écart avec la méthode B (55,0) : {mean_a - 55.0:+.2f} pp")

# À compléter : test t de Welch sur deux échantillons de graines
# from scipy import stats
# stats.ttest_ind(scores_methode_a, scores_methode_b, equal_var=False)

**Question de réflexion.** Un écart de 2 points sur cinq graines est-il convaincant ? Quels autres facteurs (taille de l'échantillon de test, variance entre graines) faut-il prendre en compte ?

## Conclusion

- Le langage naturel peut servir d'interface commune entre modalités et langues.
- Le pooling par prototypes et la pondération des classes ciblent les émotions rares.
- Les gains les plus nets sont sur les jeux de données les plus déséquilibrés, mais les petits effectifs de test demandent de la prudence.

**Pour aller plus loin :** lire la section 4.6 sur les schémas de confusion interculturels, puis comparer la figure 6 (annexe A) avec vos propres hypothèses.